In [2]:
import pandas as pd

df = pd.read_csv("superstore.csv", encoding='ISO-8859-1')

In [3]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [4]:
# 1. Sales Performance & Growth Analysis
# Generate executive summary, key findings, recommendations, Q&A, and sample records

import numpy as np
from datetime import datetime

# Executive Summary
summary = []
summary.append("Sales Performance & Growth Analysis\n")
summary.append("This section analyzes overall sales trends, top products, regional growth, and seasonality to provide actionable insights for business growth.\n")

# Key Findings & Trends
sales_by_year = df.groupby(df['Order Date'].apply(lambda x: pd.to_datetime(x).year))['Sales'].sum()
top_products = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(3)
top_regions = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)

summary.append("Key Findings & Trends:\n")
summary.append(f"- Total sales by year: {sales_by_year.to_dict()}\n")
summary.append(f"- Top 3 products by sales: {top_products.to_dict()}\n")
summary.append(f"- Sales by region: {top_regions.to_dict()}\n")

# Actionable Recommendations
summary.append("Actionable Recommendations:\n")
summary.append("- Focus marketing on top-performing regions and products.\n")
summary.append("- Investigate underperforming regions for growth opportunities.\n")
summary.append("- Leverage seasonality trends for targeted promotions.\n")

# Sample Q&A
qa = [
    ("What were the total sales in 2017?", f"${sales_by_year.get(2017, 0):,.2f}"),
    ("Which product generated the highest sales?", top_products.index[0]),
    ("Which region had the highest sales?", top_regions.index[0]),
    ("How have sales trended over the years?", str(sales_by_year.to_dict())),
    ("What are the top 3 products by sales?", ', '.join(top_products.index))
]
summary.append("Sample Q&A:\n")
for q, a in qa:
    summary.append(f"Q: {q}\nA: {a}\n")

# Sample Data Records
sample_records = df[['Order ID', 'Order Date', 'Region', 'Product Name', 'Sales']].sample(5, random_state=42)
summary.append("Sample Data Records:\n")
for _, row in sample_records.iterrows():
    summary.append(f"OrderID: {row['Order ID']}, Date: {row['Order Date']}, Region: {row['Region']}, Product: {row['Product Name']}, Sales: ${row['Sales']:.2f}\n")

# Write to TXT file
with open('Sales_Performance_and_Growth.txt', 'w', encoding='utf-8') as f:
    f.writelines(summary)

print("Sales Performance & Growth Analysis TXT file generated.")

Sales Performance & Growth Analysis TXT file generated.


In [5]:
# 2. Customer Segmentation & Behavior Analysis
# Generate customer insights, segmentation, loyalty metrics, and value analysis

customer_analysis = []
customer_analysis.append("Customer Segmentation & Behavior Analysis\n\n")
customer_analysis.append("This section analyzes customer types, loyalty, repeat purchase behavior, and customer lifetime value to support targeted retention and acquisition strategies.\n\n")

# Customer Metrics
customer_metrics = df.groupby('Customer ID').agg({
    'Sales': ['sum', 'count'],
    'Profit': 'sum'
}).reset_index()
customer_metrics.columns = ['Customer ID', 'Total Sales', 'Order Count', 'Total Profit']
avg_order_value = df.groupby('Customer ID')['Sales'].mean().mean()
repeat_customers = (customer_metrics['Order Count'] > 1).sum()
total_customers = len(customer_metrics)

customer_analysis.append("Key Findings & Trends:\n")
customer_analysis.append(f"- Total unique customers: {total_customers}\n")
customer_analysis.append(f"- Repeat customers: {repeat_customers} ({repeat_customers/total_customers*100:.1f}%)\n")
customer_analysis.append(f"- Average order value: ${avg_order_value:.2f}\n")
customer_analysis.append(f"- High-value customers (Top 10%): {int(total_customers * 0.1)}\n")
customer_analysis.append(f"- Average orders per customer: {customer_metrics['Order Count'].mean():.1f}\n\n")

customer_analysis.append("Customer Segments:\n")
customer_analysis.append(f"- By type: Consumer, Corporate, Home Office\n")
segment_dist = df['Segment'].value_counts().to_dict()
for seg, count in segment_dist.items():
    customer_analysis.append(f"  {seg}: {count} orders ({count/len(df)*100:.1f}%)\n")

customer_analysis.append("\nActionable Recommendations:\n")
customer_analysis.append("- Develop loyalty programs targeting high-value customers for retention.\n")
customer_analysis.append("- Create re-engagement campaigns for single-purchase customers.\n")
customer_analysis.append("- Offer personalized recommendations based on purchase history.\n\n")

customer_analysis.append("Sample Q&A:\n")
customer_qa = [
    ("How many unique customers do we have?", str(total_customers)),
    ("What percentage of customers are repeat buyers?", f"{repeat_customers/total_customers*100:.1f}%"),
    ("What is the average order value?", f"${avg_order_value:.2f}"),
    ("How many orders does the average customer place?", f"{customer_metrics['Order Count'].mean():.1f}"),
    ("Which customer segment is the largest?", f"Consumer with {segment_dist.get('Consumer', 0)} orders")
]
for q, a in customer_qa:
    customer_analysis.append(f"Q: {q}\nA: {a}\n")

customer_analysis.append("\nSample Customer Records:\n")
sample_customers = df[['Customer ID', 'Customer Name', 'Segment', 'Sales', 'Order Date']].drop_duplicates('Customer ID').sample(5, random_state=42)
for _, row in sample_customers.iterrows():
    customer_analysis.append(f"CustomerID: {row['Customer ID']}, Name: {row['Customer Name']}, Segment: {row['Segment']}, Sales: ${row['Sales']:.2f}\n")

with open('Customer_Segmentation_and_Behavior.txt', 'w', encoding='utf-8') as f:
    f.writelines(customer_analysis)

print("Customer Segmentation & Behavior Analysis TXT file generated.")

Customer Segmentation & Behavior Analysis TXT file generated.


In [6]:
# 3. Profitability & Cost Optimization Analysis
# Analyze profit trends, losses, discount impact, and profitability by product/category

profit_analysis = []
profit_analysis.append("Profitability & Cost Optimization Analysis\n\n")
profit_analysis.append("This section analyzes profit drivers, loss-making products, discount impact, and opportunities for margin optimization.\n\n")

# Profitability Metrics
total_profit = df['Profit'].sum()
total_sales = df['Sales'].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
loss_orders = (df['Profit'] < 0).sum()
profitable_orders = (df['Profit'] >= 0).sum()

profit_by_category = df.groupby('Category')['Profit'].sum().sort_values(ascending=False)
loss_products = df.groupby('Product Name')['Profit'].sum().sort_values().head(5)

profit_analysis.append("Key Findings & Trends:\n")
profit_analysis.append(f"- Total profit: ${total_profit:,.2f}\n")
profit_analysis.append(f"- Total sales: ${total_sales:,.2f}\n")
profit_analysis.append(f"- Overall profit margin: {profit_margin:.2f}%\n")
profit_analysis.append(f"- Loss-making orders: {loss_orders} ({loss_orders/len(df)*100:.1f}%)\n")
profit_analysis.append(f"- Profitable orders: {profitable_orders} ({profitable_orders/len(df)*100:.1f}%)\n")
profit_analysis.append(f"- Profit by category: {profit_by_category.to_dict()}\n\n")

profit_analysis.append("Top Loss-Making Products:\n")
for prod, profit in loss_products.items():
    profit_analysis.append(f"- {prod}: ${profit:,.2f}\n")

# Discount Analysis
avg_discount = df['Discount'].mean()
high_discount_loss = df[df['Discount'] > 0.3]['Profit'].mean()
no_discount_profit = df[df['Discount'] == 0]['Profit'].mean()

profit_analysis.append(f"\nDiscount Impact Analysis:\n")
profit_analysis.append(f"- Average discount: {avg_discount*100:.1f}%\n")
profit_analysis.append(f"- Avg profit with high discount (>30%): ${high_discount_loss:,.2f}\n")
profit_analysis.append(f"- Avg profit with no discount: ${no_discount_profit:,.2f}\n\n")

profit_analysis.append("Actionable Recommendations:\n")
profit_analysis.append("- Eliminate or reprice loss-making products immediately.\n")
profit_analysis.append("- Reduce excessive discounts to improve margins.\n")
profit_analysis.append("- Focus on high-margin categories for promotional budget.\n\n")

profit_analysis.append("Sample Q&A:\n")
profit_qa = [
    ("What is our overall profit margin?", f"{profit_margin:.2f}%"),
    ("How many orders are unprofitable?", str(loss_orders)),
    ("What is the impact of high discounts on profit?", f"${high_discount_loss:,.2f} vs ${no_discount_profit:,.2f} (no discount)"),
    ("Which category is most profitable?", f"{profit_by_category.index[0]} with ${profit_by_category.iloc[0]:,.2f}"),
    ("What percentage of orders are profitable?", f"{profitable_orders/len(df)*100:.1f}%")
]
for q, a in profit_qa:
    profit_analysis.append(f"Q: {q}\nA: {a}\n")

profit_analysis.append("\nSample Data Records (Loss-Making Orders):\n")
loss_sample = df[df['Profit'] < 0][['Order ID', 'Product Name', 'Sales', 'Discount', 'Profit']].sample(min(5, len(df[df['Profit'] < 0])), random_state=42)
for _, row in loss_sample.iterrows():
    profit_analysis.append(f"OrderID: {row['Order ID']}, Product: {row['Product Name']}, Sales: ${row['Sales']:.2f}, Discount: {row['Discount']*100:.0f}%, Profit: ${row['Profit']:.2f}\n")

with open('Profitability_and_Cost_Optimization.txt', 'w', encoding='utf-8') as f:
    f.writelines(profit_analysis)

print("Profitability & Cost Optimization Analysis TXT file generated.")

Profitability & Cost Optimization Analysis TXT file generated.


In [7]:
# 4. Supply Chain & Operations Efficiency Analysis
# Analyze shipping performance, delivery times, and operational metrics

supply_analysis = []
supply_analysis.append("Supply Chain & Operations Efficiency Analysis\n\n")
supply_analysis.append("This section analyzes shipping modes, delivery performance, and operational efficiency to optimize fulfillment processes.\n\n")

# Shipping Mode Analysis
shipping_dist = df['Ship Mode'].value_counts()
shipping_profit = df.groupby('Ship Mode')['Profit'].mean()
shipping_time = df.groupby('Ship Mode').apply(
    lambda x: (pd.to_datetime(x['Ship Date']) - pd.to_datetime(x['Order Date'])).dt.days.mean()
)

supply_analysis.append("Key Findings & Trends:\n")
supply_analysis.append("Shipping Mode Distribution:\n")
for mode, count in shipping_dist.items():
    supply_analysis.append(f"- {mode}: {count} orders ({count/len(df)*100:.1f}%)\n")

supply_analysis.append("\nAverage Shipping Time by Mode:\n")
for mode, days in shipping_time.items():
    supply_analysis.append(f"- {mode}: {days:.1f} days\n")

supply_analysis.append("\nAverage Profit by Shipping Mode:\n")
for mode, profit in shipping_profit.items():
    supply_analysis.append(f"- {mode}: ${profit:,.2f}\n")

# Regional Logistics
regional_orders = df.groupby('Region')['Order ID'].count()
regional_shipping = df.groupby('Region')['Ship Mode'].value_counts().unstack(fill_value=0)

supply_analysis.append("\nRegional Order Volume:\n")
for region, count in regional_orders.items():
    supply_analysis.append(f"- {region}: {count} orders\n")

supply_analysis.append("\nActionable Recommendations:\n")
supply_analysis.append("- Prioritize faster shipping modes for high-value orders.\n")
supply_analysis.append("- Optimize regional warehousing to reduce shipping times.\n")
supply_analysis.append("- Balance shipping cost vs. profit margin for each mode.\n\n")

supply_analysis.append("Sample Q&A:\n")
supply_qa = [
    ("Which shipping mode is most commonly used?", f"{shipping_dist.index[0]} with {shipping_dist.iloc[0]} orders"),
    ("What is the average shipping time?", f"{shipping_time.mean():.1f} days"),
    ("Which region has the most orders?", f"{regional_orders.idxmax()} with {regional_orders.max()} orders"),
    ("Does faster shipping improve profitability?", "Variable; Standard Class averages " + f"${shipping_profit.get('Standard Class', 0):.2f}"),
    ("What is the average shipping time for Same Day delivery?", f"{shipping_time.get('Same Day', 0):.1f} days")
]
for q, a in supply_qa:
    supply_analysis.append(f"Q: {q}\nA: {a}\n")

supply_analysis.append("\nSample Data Records (Orders by Region & Shipping):\n")
sample_supply = df[['Order ID', 'Region', 'Ship Mode', 'Order Date', 'Ship Date']].sample(5, random_state=42)
for _, row in sample_supply.iterrows():
    days = (pd.to_datetime(row['Ship Date']) - pd.to_datetime(row['Order Date'])).days
    supply_analysis.append(f"OrderID: {row['Order ID']}, Region: {row['Region']}, Mode: {row['Ship Mode']}, Processing Time: {days} days\n")

with open('Supply_Chain_and_Operations_Efficiency.txt', 'w', encoding='utf-8') as f:
    f.writelines(supply_analysis)

print("Supply Chain & Operations Efficiency Analysis TXT file generated.")

Supply Chain & Operations Efficiency Analysis TXT file generated.


C:\Users\SHANKHADEEP\AppData\Local\Temp\ipykernel_9792\2219094454.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  shipping_time = df.groupby('Ship Mode').apply(


In [8]:
# 5. Market & Regional Insights Analysis
# Analyze geographic performance, regional trends, and market opportunities

market_analysis = []
market_analysis.append("Market & Regional Insights Analysis\n\n")
market_analysis.append("This section provides geographic performance analysis, regional trends, and identifies growth opportunities and market risks by region and state.\n\n")

# Regional Performance
regional_sales = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).rename(columns={'Order ID': 'Orders'})
regional_sales['Profit Margin %'] = (regional_sales['Profit'] / regional_sales['Sales'] * 100).round(2)
regional_sales = regional_sales.sort_values('Sales', ascending=False)

state_performance = df.groupby('State').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).rename(columns={'Order ID': 'Orders'}).sort_values('Sales', ascending=False).head(10)

market_analysis.append("Key Findings & Trends:\n")
market_analysis.append("Regional Performance Summary:\n")
for region, row in regional_sales.iterrows():
    market_analysis.append(f"- {region}: Sales ${row['Sales']:,.0f}, Profit ${row['Profit']:,.0f}, Margin {row['Profit Margin %']:.1f}%, Orders {int(row['Orders'])}\n")

market_analysis.append("\nTop 10 States by Sales:\n")
for state, row in state_performance.iterrows():
    market_analysis.append(f"- {state}: ${row['Sales']:,.0f} sales, {int(row['Orders'])} orders\n")

# State Risk Analysis (Unprofitable States)
state_profit = df.groupby('State')['Profit'].sum().sort_values()
unprofitable_states = state_profit[state_profit < 0].head(5)

market_analysis.append("\nUnprofitable States (Risk Areas):\n")
for state, profit in unprofitable_states.items():
    market_analysis.append(f"- {state}: ${profit:,.0f}\n")

# Category Performance by Region
category_regional = df.pivot_table(values='Sales', index='Region', columns='Category', aggfunc='sum')

market_analysis.append("\nCategory Performance by Region:\n")
for region in category_regional.index:
    market_analysis.append(f"- {region}: Technology ${category_regional.loc[region, 'Technology']:,.0f}, Furniture ${category_regional.loc[region, 'Furniture']:,.0f}, Office Supplies ${category_regional.loc[region, 'Office Supplies']:,.0f}\n")

market_analysis.append("\nActionable Recommendations:\n")
market_analysis.append("- Expand operations in high-performing regions (East, West).\n")
market_analysis.append("- Investigate and turnaround unprofitable states with targeted interventions.\n")
market_analysis.append("- Adjust product mix by region to match local demand.\n\n")

market_analysis.append("Sample Q&A:\n")
market_qa = [
    ("Which region has the highest sales?", f"{regional_sales.index[0]} with ${regional_sales.iloc[0]['Sales']:,.0f}"),
    ("Which region is most profitable?", f"{regional_sales['Profit'].idxmax()} with ${regional_sales['Profit'].max():,.0f}"),
    ("Which states are unprofitable?", ', '.join(unprofitable_states.index.tolist())),
    ("What is the profit margin by region?", "East: " + f"{regional_sales.loc['East', 'Profit Margin %']:.1f}%, " + "West: " + f"{regional_sales.loc['West', 'Profit Margin %']:.1f}%"),
    ("Which is the top state by sales?", f"{state_performance.index[0]} with ${state_performance.iloc[0]['Sales']:,.0f}")
]
for q, a in market_qa:
    market_analysis.append(f"Q: {q}\nA: {a}\n")

market_analysis.append("\nSample Data Records (Regional Orders):\n")
sample_market = df[['Order ID', 'State', 'Region', 'Sales', 'Profit']].sample(5, random_state=42)
for _, row in sample_market.iterrows():
    market_analysis.append(f"OrderID: {row['Order ID']}, State: {row['State']}, Region: {row['Region']}, Sales: ${row['Sales']:.2f}, Profit: ${row['Profit']:.2f}\n")

with open('Market_and_Regional_Insights.txt', 'w', encoding='utf-8') as f:
    f.writelines(market_analysis)

print("Market & Regional Insights Analysis TXT file generated.")

Market & Regional Insights Analysis TXT file generated.
